## notebook 02
- parsing and cleaning raw protacdb data for model-ready data but also preserve original raw data
  
goals:
- parse original dc50 and dmax measurements
- canonicalise protac smiles w rdkit 
- identify invalid structures and chemical duplicates
- define rules for repeated measurements
- define the degradation label
- export the processed dataset

In [1]:
from pathlib import Path

import pandas as pd
from rdkit import Chem
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from parsing import parse_measurement

### 0. load protac db raw data

In [2]:
data_path = project_root / "data" / "raw" / "protac_db.csv"

protacdb_df = pd.read_csv(data_path)
protacdb_df.shape

/var/folders/mt/fkwf2zrd2yg6bnj3dhg3flph0000gn/T/ipykernel_56843/8954242.py:3: DtypeWarning: Columns (14: EC50 (nM, Protac to Target), 20: delta G (kcal/mol, Protac to Target), 21: delta H (kcal/mol, Protac to Target), 22: -T*delta S (kcal/mol, Protac to Target), 35: Assay (Protac to E3, Ki), 44: IC50 (nM, Ternary complex), 46: EC50 (nM, Ternary complex), 50: Ki (nM, Ternary complex), 52: delta G (kcal/mol, Ternary complex), 53: delta H (kcal/mol, Ternary complex), 54: -T*delta S (kcal/mol, Ternary complex), 56: kon (1/Ms, Ternary complex), 57: koff (1/s, Ternary complex), 68: GR50 (nM, Cellular activities), 69: Assay (Cellular activities, GR50)) have mixed types. Specify dtype option on import or set low_memory=False.
  protacdb_df = pd.read_csv(data_path)


(15502, 89)

### 1. parse raw protacdb data

In [3]:
dc50_parsed = protacdb_df["DC50 (nM)"].apply(parse_measurement)
dmax_parsed = protacdb_df["Dmax (%)"].apply(parse_measurement)

dc50_types = dc50_parsed.apply(lambda x: x["type"]).value_counts()
dmax_types = dmax_parsed.apply(lambda x: x["type"]).value_counts()

type_counts = pd.concat(
    [dc50_types, dmax_types],
    axis=1,
    keys=["DC50", "Dmax"]
).fillna(0).astype(int)

type_counts

,DC50,Dmax
missing,12001,13177
exact,2277,1586
lower_bound,402,327
upper_bound,399,41
multiple,360,216
range,60,153
unparsed,3,2


In [4]:
dc50_unparsed = dc50_parsed[dc50_parsed.apply(lambda x: x["type"]) == "unparsed"]
dmax_unparsed = dmax_parsed[dmax_parsed.apply(lambda x: x["type"]) == "unparsed"]

display(dc50_unparsed.value_counts())
display(dmax_unparsed.value_counts())

DC50 (nM)
{'type': 'unparsed', 'raw': '51.8 82.5 70.2', 'lower': None, 'upper': None}    1
{'type': 'unparsed', 'raw': '21.2 31.2 25.3', 'lower': None, 'upper': None}    1
{'type': 'unparsed', 'raw': '1630(nm)', 'lower': None, 'upper': None}          1
Name: count, dtype: int64

Dmax (%)
{'type': 'unparsed', 'raw': '2000-06-18 00:00:00', 'lower': None, 'upper': None}    1
{'type': 'unparsed', 'raw': '2000-11-06 00:00:00', 'lower': None, 'upper': None}    1
Name: count, dtype: int64

In [5]:
# #round one parsing: bottom row had no type
# blank_type_mask = dmax_parsed.apply(lambda x: x["type"]) == ""

# protacdb_df.loc[blank_type_mask, "Dmax (%)"]

In [6]:
# dmax_with_hyphen = protacdb_df.loc[
#     protacdb_df["Dmax (%)"].astype(str).str.count("-") > 1,
#     "Dmax (%)"
# ]

# dmax_with_hyphen.value_counts()

# # found multiple hyphens in dmax values, which are dates as shown. we will only parse ranges if there is exactly one hyphen (reflected in parsing.py)

findings: 
- 5 measurements unparsed:
  - 3 dc50 values:
    - `51.8 82.5 70.2`
    - `21.2 31.2 25.3`
    - `1630(nm)`
    - cannot assume values separated by " " are same assays, leave unparsed
    - only one value with (nm) so leave unparsed
  - 2 dmax values:
    - `2000-06-18 00:00:00`
    - `2000-11-06 00:00:00`
    - likely input error, leave unparsed

### 2. canonicalise protac smiles string with rdkit
1. tested with first smiles first

In [7]:
example_smiles = protacdb_df["Smiles"].dropna().iloc[0]
mol = Chem.MolFromSmiles(example_smiles)
display(example_smiles)
display(mol)

'COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN1CCN(CCOCCOCC(=O)N[C@H](C(=O)N2C[C@H](O)C[C@H]2C(=O)NCC2=CC=C(C3=C(C)N=CS3)C=C2)C(C)(C)C)CC1'

In [8]:
canonicalised_smiles = Chem.MolToSmiles(mol)

print("original:")
print(example_smiles)

print("\ncanonical:")
print(canonicalised_smiles)

original:
COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN1CCN(CCOCCOCC(=O)N[C@H](C(=O)N2C[C@H](O)C[C@H]2C(=O)NCC2=CC=C(C3=C(C)N=CS3)C=C2)C(C)(C)C)CC1

canonical:
COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(CCOCCOCC(=O)N[C@H](C(=O)N2C[C@H](O)C[C@H]2C(=O)NCc2ccc(-c3scnc3C)cc2)C(C)(C)C)CC1


2. canonicalise function to apply to whole pd

In [9]:
def canonicalise_smiles(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

protacdb_df["Canonicalised Smiles"] = protacdb_df["Smiles"].apply(canonicalise_smiles)

print("total number of rows:", len(protacdb_df))
print("missing raw smiles", protacdb_df["Smiles"].isna().sum())
print("missing canonicalised smiles", protacdb_df["Canonicalised Smiles"].isna().sum())

total number of rows: 15502
missing raw smiles 0
missing canonicalised smiles 0


In [10]:
print("unique raw smiles:", protacdb_df["Smiles"].nunique())
print("unique canonicalised smiles:", protacdb_df["Canonicalised Smiles"].nunique())

structures_per_compoundid = protacdb_df.groupby("Compound ID")["Canonicalised Smiles"].nunique()
print("number of canonicalised structures per compound id:", structures_per_compoundid.value_counts().sort_index())

unique raw smiles: 10728
unique canonicalised smiles: 10728
number of canonicalised structures per compound id: Canonicalised Smiles
1    10722
2        3
Name: count, dtype: int64


In [11]:
multi_structure_ids = structures_per_compoundid[structures_per_compoundid > 1].index
multi_structure_ids

Index([4151, 5363, 5369], dtype='int64', name='Compound ID')

In [12]:
protacdb_df.loc[
    protacdb_df["Compound ID"].isin(multi_structure_ids), 
[
    "Compound ID", "Name", "Smiles", "Canonicalised Smiles", "Target", "E3 ligase", "Article DOI"
    ]
].sort_values("Compound ID")

,Compound ID,Name,Smiles,Canonicalised Smiles,Target,E3 ligase,Article DOI
6850,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,cIAP1,CRBN,10.1021/acs.jmedchem.2c01817
6851,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,cIAP2,CRBN,10.1021/acs.jmedchem.2c01817
6852,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,XIAP,CRBN,10.1021/acs.jmedchem.2c01817
6853,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,CRBN,cIAP1,10.1021/acs.jmedchem.2c01817
6854,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,cIAP1,CRBN,10.1021/acs.jmedchem.2c01817
6855,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,cIAP2,CRBN,10.1021/acs.jmedchem.2c01817
6856,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,XIAP,CRBN,10.1021/acs.jmedchem.2c01817
6857,4151,NaN,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](OC2=CC=C...,CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(...,CRBN,cIAP1,10.1021/acs.jmedchem.2c01817
8616,5363,NaN,O=C1CCC(N2C(=O)C3=CC(F)=C(N4CCC(CN5CCC(CCC(=O)...,O=C1CCC(N2C(=O)c3cc(F)c(N4CCC(CN5CCC(CCC(=O)Nc...,USP7,CRBN,10.1039/d2cc02094g
8617,5363,NaN,C[C@H](CC(=O)N1CCC(O)(CN2C=NC3=CC(NC(=O)CCC4CC...,C[C@H](CC(=O)N1CCC(O)(Cn2cnc3cc(NC(=O)CCC4CCN(...,USP7,CRBN,10.1039/d2cc02094g


to fix/investigate:
- 5363: same compound id, two different molecular structures
- 4151, 5369: unclear how structures differ

In [13]:
for compound_id in multi_structure_ids:
    structures = (
        protacdb_df.loc[
            protacdb_df["Compound ID"] == compound_id,
            "Canonicalised Smiles"
        ]
    .drop_duplicates().tolist()
    )

    print(f"compound id: {compound_id}")

    for structure in structures:
        print(structure)

compound id: 4151
CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(OCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c2)C[C@H]1C(=O)N[C@@H]1CCCc2ccccc21)C1CCCCC1
CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(OCCCCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c2)C[C@H]1C(=O)N[C@@H]1CCCc2ccccc21)C1CCCCC1
compound id: 5363
O=C1CCC(N2C(=O)c3cc(F)c(N4CCC(CN5CCC(CCC(=O)Nc6ccc7c(=O)n(CC8(O)CCN(C(=O)CC(c9ccccc9)C(F)(F)F)CC8)cnc7c6)CC5)CC4)cc3C2=O)C(=O)N1
C[C@H](CC(=O)N1CCC(O)(Cn2cnc3cc(NC(=O)CCC4CCN(CC5CCN(c6cc7c(cc6F)C(=O)N(C6CCC(=O)NC6=O)C7=O)CC5)CC4)ccc3c2=O)CC1)c1ccccc1
compound id: 5369
CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4ccc(C(=O)NCCOCCOCCOCCOc5cc(-c6scnc6C)ccc5CNC(=O)[C@@H]5C[C@@H](O)CN5C(=O)[C@H](C(C)C)N5Cc6ccccc6C5=O)cc4)cc23)c1F
CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4ccc(CNC(=O)CCCCOCCOCCOCCOc5cc(-c6scnc6C)ccc5CNC(=O)[C@@H]5C[C@@H](O)CN5C(=O)[C@H](C(C)C)N5Cc6ccccc6C5=O)cc4)cc23)c1F


#### canonicalised smiles summary
- all 15,502 smiles successfully canonicalised
- total of 10728 unique raw smiles, and 10728 unique canonicalised smiles :)
- coumpound ids 4151, 5363, 5369 contain multiple molecular structures 
  - use smiles strings as molecular identity instead of compound ids
  - retain compound id as source metadata

### 3. creating degradation labels
- labeling our cleaned data as training data for ML model

In [14]:
protacdb_df["DC50 Parsed"] = dc50_parsed
protacdb_df["Dmax Parsed"] = dmax_parsed

usable_types = {"exact", "lower_bound", "upper_bound", "range", "multiple"}

protacdb_df["DC50 type"] = protacdb_df["DC50 Parsed"].str.get("type")
dc50_usable = protacdb_df["DC50 type"].isin(usable_types)

protacdb_df["Dmax type"] = protacdb_df["Dmax Parsed"].str.get("type")
dmax_usable = protacdb_df["Dmax type"].isin(usable_types)

print("usable DC50:", dc50_usable.sum())
print("usable Dmax:", dmax_usable.sum())
print("usable both:", (dc50_usable & dmax_usable).sum())
print("usable either:", (dc50_usable | dmax_usable).sum())

usable DC50: 3498
usable Dmax: 2323
usable both: 2157
usable either: 3664


comparing two literature-grounded activity definitions before selecting the final labeling strategy:

1. **se(3)-protacs** — kothakapu et al. (2026), *SE(3)-PROTACs: Geometric deep learning for PROTAC degradation prediction*. **Briefings in Bioinformatics, 27(3), bbag228.** https://doi.org/10.1093/bib/bbag228  
   - good degrader: **DC50 ≤ 100 nM AND Dmax ≥ 80%**

2. **pandiyan et al.** — pandiyan et al. (2026), *Predicting PROTAC degradation activity and selectivity of effective E3 ligase through harnessing a combination of AtomPair fingerprints and multiple machine learning algorithms*. **Journal of Molecular Graphics and Modelling, 146, 109449.** https://doi.org/10.1016/j.jmgm.2026.109449  
   - active PROTAC criterion: **DC50 < 10 μM OR Dmax > 30%**

In [30]:
from labeling_se3 import evaluate_dc50, evaluate_dmax, label_se3

# calculate SE(3) measurement statuses
protacdb_df["SE3 DC50 Status"] = protacdb_df["DC50 Parsed"].apply(evaluate_dc50, args=(100,))

protacdb_df["SE3 Dmax Status"] = protacdb_df["Dmax Parsed"].apply(evaluate_dmax, args=(80,))

# calculate final label AFTER recalculating statuses
protacdb_df["SE3 Label"] = protacdb_df.apply(lambda row: label_se3(row["SE3 DC50 Status"], row["SE3 Dmax Status"]),axis=1)

# inspect final label distribution
print("all rows:")
print(protacdb_df["SE3 Label"].value_counts())

# create labeled subset
se3_labeled = protacdb_df[protacdb_df["SE3 Label"] != "ambiguous"].copy()

print("\nlabeled rows:")
print(se3_labeled["SE3 Label"].value_counts())

print("\nclass balance:")
print(se3_labeled["SE3 Label"].value_counts(normalize=True))

print("\ntotal labeled rows:", len(se3_labeled))
print("unique molecules:", se3_labeled["Canonicalised Smiles"].nunique())
print("unique uniprot targets:", se3_labeled["Uniprot"].nunique())

all rows:
SE3 Label
ambiguous    13165
negative      1533
positive       804
Name: count, dtype: int64

labeled rows:
SE3 Label
negative    1533
positive     804
Name: count, dtype: int64

class balance:
SE3 Label
negative    0.655969
positive    0.344031
Name: proportion, dtype: float64

total labeled rows: 2337
unique molecules: 1824
unique uniprot targets: 156


In [31]:
#2. pandiyan et al thresholds

from labeling_pandiyan import evaluate_dc50, evaluate_dmax, label_pandiyan

# evaluate dc50 < 10,000 nm and dmax > 30%
protacdb_df["Pandiyan DC50 Status"] = protacdb_df["DC50 Parsed"].apply(evaluate_dc50, args=(10000,))

protacdb_df["Pandiyan Dmax Status"] = protacdb_df["Dmax Parsed"].apply(evaluate_dmax, args=(30,))

# combine using pandiyan's OR rule
protacdb_df["Pandiyan Label"] = protacdb_df.apply(lambda row: label_pandiyan(row["Pandiyan DC50 Status"], row["Pandiyan Dmax Status"]),axis=1)

# inspect overall label distribution
print("all rows:")
print(protacdb_df["Pandiyan Label"].value_counts())
print(protacdb_df["Pandiyan Label"].value_counts(normalize=True))

# create labeled subset
pandiyan_labeled = protacdb_df[protacdb_df["Pandiyan Label"] != "ambiguous"].copy()

# inspect modeling subset
print("\nlabeled rows:")
print(pandiyan_labeled["Pandiyan Label"].value_counts())

print("\nclass balance:")
print(pandiyan_labeled["Pandiyan Label"].value_counts(normalize=True))

print("\ntotal labeled rows:", len(pandiyan_labeled))
print("unique molecules:", pandiyan_labeled["Canonicalised Smiles"].nunique())
print("unique uniprot targets:", pandiyan_labeled["Uniprot"].nunique())

all rows:
Pandiyan Label
ambiguous    12614
positive      2872
negative        16
Name: count, dtype: int64
Pandiyan Label
ambiguous    0.813701
positive     0.185266
negative     0.001032
Name: proportion, dtype: float64

labeled rows:
Pandiyan Label
positive    2872
negative      16
Name: count, dtype: int64

class balance:
Pandiyan Label
positive    0.99446
negative    0.00554
Name: proportion, dtype: float64

total labeled rows: 2888
unique molecules: 2298
unique uniprot targets: 182


results of labeling definitions:
- se(3) definition produces a highly negative-skewed dataset
  - labeled rows: 2,824
  - positive: 87 (3.1%)
  - negative: 2,737 (96.9%)
- pandiyan definition produces a highly positive-skewed dataset
  - labeled rows: 2,888
  - positive: 2,872 (99.4%)
  - negative: 16 (0.6%)
- neither published definition produces a sufficiently balanced binary dataset as-is

labeling definitions tested:
- se(3): DC50 ≤ 100 nM AND Dmax ≥ 80%
- pandiyan: DC50 < 10,000 nM OR Dmax > 30%

next step:
- do not select a final label definition yet
- use se(3) and pandiyan as stringent and permissive reference points
- inspect the DC50/Dmax distributions and determine a biologically defensible labeling strategy appropriate for this dataset

### 3.5 testing third literature reference threshold 
ribes et al. (2024), modeling protac degradation activity with machine learning defines an active protac as:
- pDC50 ≥ 6, equivalent to DC50 ≤ 1,000 nM
- Dmax ≥ 60%
- both conditions must be satisfied (AND)

this has the same ≤ DC50 AND ≥ Dmax logic as se(3), so we reuse the se(3) labeling functions and change only the thresholds

In [32]:
# evaluate individual threshold conditions
protacdb_df["Ribes DC50 Status"] = protacdb_df["DC50 Parsed"].apply(
    evaluate_dc50,
    args=(1000,)
)

protacdb_df["Ribes Dmax Status"] = protacdb_df["Dmax Parsed"].apply(
    evaluate_dmax,
    args=(60,)
)

# combine using the same AND logic as se(3)
protacdb_df["Ribes Label"] = protacdb_df.apply(
    lambda row: label_se3(
        row["Ribes DC50 Status"],
        row["Ribes Dmax Status"]
    ),
    axis=1
)

# overall label distribution
print("all rows:")
print(protacdb_df["Ribes Label"].value_counts())
print(protacdb_df["Ribes Label"].value_counts(normalize=True))

# exclude ambiguous observations
ribes_labeled = protacdb_df[
    protacdb_df["Ribes Label"] != "ambiguous"
].copy()

# inspect usable binary dataset
print("\nlabeled rows:")
print(ribes_labeled["Ribes Label"].value_counts())

print("\nclass balance:")
print(ribes_labeled["Ribes Label"].value_counts(normalize=True))

print("\ntotal labeled rows:", len(ribes_labeled))
print("unique molecules:", ribes_labeled["Canonicalised Smiles"].nunique())
print("unique uniprot targets:", ribes_labeled["Uniprot"].nunique())

all rows:
Ribes Label
ambiguous    13248
positive      1436
negative       818
Name: count, dtype: int64
Ribes Label
ambiguous    0.854599
positive     0.092633
negative     0.052767
Name: proportion, dtype: float64

labeled rows:
Ribes Label
positive    1436
negative     818
Name: count, dtype: int64

class balance:
Ribes Label
positive    0.63709
negative    0.36291
Name: proportion, dtype: float64

total labeled rows: 2254
unique molecules: 1782
unique uniprot targets: 143


#### results of labeling definition:
- selected ribes thresholds: literature-based, with moderate class imbalance (63.7% positive / 36.3% negative).
- labeled dataset: 2,254 rows; 1,436 positive and 818 negative.
- positive (`1`): dc50 ≤ 1000 nm **and** dmax ≥ 60%.
- negative (`0`): either criterion confidently fails.
- ambiguous: insufficient evidence; excluded from modeling.

In [33]:
# create final modeling dataset using ribes labels

protac_clean = protacdb_df[
    protacdb_df["Ribes Label"] != "ambiguous"
].copy()

# convert labels to binary values for modeling
protac_clean["Label"] = protac_clean["Ribes Label"].map({
    "negative": 0,
    "positive": 1
})

# verify final labels
print("shape:", protac_clean.shape)

print("\nlabel distribution:")
print(protac_clean["Label"].value_counts())

print("\nclass balance:")
print(protac_clean["Label"].value_counts(normalize=True))

print("\nunique molecules:", protac_clean["Canonicalised Smiles"].nunique())
print("unique uniprot targets:", protac_clean["Uniprot"].nunique())

shape: (2254, 104)

label distribution:
Label
1    1436
0     818
Name: count, dtype: int64

class balance:
Label
1    0.63709
0    0.36291
Name: proportion, dtype: float64

unique molecules: 1782
unique uniprot targets: 143


In [34]:
# save two final csvs

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# create the relevant-row subset from ribes labels
model_columns = [
    "Compound ID",
    "Article DOI",
    "Canonicalised Smiles",
    "Target",
    "Uniprot",
    "E3 ligase",
    "DC50 (nM)",
    "Dmax (%)",
    "Assay (DC50/Dmax)",
    "Ribes Label",
]

protac_clean = protacdb_df.loc[
    protacdb_df["Ribes Label"].isin(["positive", "negative"]),
    model_columns
].copy()

protac_clean["Label"] = protac_clean["Ribes Label"].map({
    "negative": 0,
    "positive": 1
}).astype(int)

# save all rows with all parsing/cleaning columns
protacdb_df.to_csv(processed_dir / "protacdb_df.csv", index=False)

# save relevant rows with ribes labels
protac_clean.to_csv(processed_dir / "protac_clean.csv", index=False)

print("protacdb_df.csv:", protacdb_df.shape)
print("protac_clean.csv:", protac_clean.shape)
print("\nribes label counts:")
print(protac_clean["Label"].value_counts())

protacdb_df.csv: (15502, 104)
protac_clean.csv: (2254, 11)

ribes label counts:
Label
1    1436
0     818
Name: count, dtype: int64


## summary: parsing, structure cleaning, and labeling

- parsed dc50/dmax into exact values, bounds, ranges, multiple measurements, missing, or unparsed.
- preserved raw values; did not average multiple measurements.
- left 5 unsupported cells unparsed: 3 dc50 values and 2 date-like dmax values.
- all 15,502 smiles parsed with rdkit; 10,728 unique canonical structures.
- compound ids 4151, 5363, and 5369 each map to two structures; use canonical smiles as molecular identity.

### label selection

- compared three literature-based definitions before training.
- selected ribes: dc50 ≤ 1000 nm **and** dmax ≥ 60%.
- positive (`1`): both criteria confidently pass.
- negative (`0`): either criterion confidently fails; does not necessarily mean no degradation.
- ambiguous: insufficient evidence, including unresolved multiple measurements; excluded from modeling.
- resulting subset: **2,254 rows**, **1,782 molecules**, **143 uniprot targets**.
- class balance: **1,436 positive (63.7%) / 818 negative (36.3%)**.

### outputs

- `data/processed/protacdb_df.csv`: all 15,502 rows with parsing, cleaning, and candidate labels.
- `data/processed/protac_clean.csv`: relevant columns and ribes labels for 2,254 observations.
- next: begin day 4 from `protac_clean.csv`; target mapping and split checks are still pending.